# pyens — Core Ensemble Spec

This notebook walks through the Phase 1 implementation: the structured input algebra for specifying ensemble model runs.

The three core ideas:
1. **`Axis`** — a named dimension of the ensemble coordinate space.
2. **`Fixed` / `Grid`** — field specs describing how each model input varies (or doesn't) across runs.
3. **`EnsembleSpec`** — combines fields into a full ensemble; produces concrete input dicts for each run.

The algebra rule that governs everything:
> Two `Grid` fields on the **same `Axis` instance** co-vary (zip). Two fields on **different `Axis` instances** are crossed (Cartesian product).

In [1]:
from pyens import Axis, Fixed, Grid, EnsembleSpec, BoundSpec

---
## 1. Axis

An `Axis` names a dimension and defines its labels. You can provide explicit labels or just a count.

In [2]:
# Named labels — useful for sites, species, experiments, etc.
sites = Axis("site", labels=["harvard_forest", "niwot_ridge", "metolius"])
print(sites)
print(f"size: {sites.size}")
print(f"label_at(1): {sites.label_at(1)}")

Axis('site', labels=['harvard_forest', 'niwot_ridge', 'metolius'])
size: 3
label_at(1): niwot_ridge


In [3]:
# Integer-labeled — useful for ensemble members, parameter sets, etc.
members = Axis("member", size=5)
print(members)
print(f"labels: {members.labels}")

Axis('member', size=5)
labels: (0, 1, 2, 3, 4)


**Identity semantics:** Two `Axis` objects are the *same dimension* only if they are the same Python object — not just if they share a name. This is what controls zip vs. Cartesian product below.

In [4]:
ax_a = Axis("x", size=3)
ax_b = Axis("x", size=3)   # same name, different object

print(f"ax_a is ax_a: {ax_a is ax_a}")  # True — same dimension
print(f"ax_a is ax_b: {ax_a is ax_b}")  # False — different dimension

ax_a is ax_a: True
ax_a is ax_b: False


---
## 2. Field Specs: `Fixed` and `Grid`

Each named input to the model is described by a `FieldSpec`.

### `Fixed` — same value every run

In [5]:
base_params = {"aMax": 11.0, "psnTOpt": 20.0}  # pretend SIPNET-like parameters
f = Fixed(base_params)

print(f)
print(f"axes: {f.axes}")               # empty — Fixed contributes no dimension
print(f"value_at({{}}): {f.value_at({})}")  # same value regardless of coords

Fixed({'aMax': 11.0, 'psnTOpt': 20.0})
axes: ()
value_at({}): {'aMax': 11.0, 'psnTOpt': 20.0}


### `Grid` — value indexed along one or more axes

In [6]:
# One climate dict per site — a single-axis Grid
climate_data = [
    {"tair_mean": 8.5},   # harvard_forest
    {"tair_mean": 1.2},   # niwot_ridge
    {"tair_mean": 9.0},   # metolius
]
climate = Grid(climate_data, along=sites)

print(climate)
print(f"axes: {climate.axes}")

# Look up value at index 1 (niwot_ridge)
print(f"value at site=1: {climate.value_at({sites: 1})}")

Grid(<3 values>, along=Axis('site', labels=['harvard_forest', 'niwot_ridge', 'metolius']))
axes: (Axis('site', labels=['harvard_forest', 'niwot_ridge', 'metolius']),)
value at site=1: {'tair_mean': 1.2}


In [7]:
# Multi-axis Grid: IC (initial plant wood C, g C/m²) varies across both sites AND members
# Shape: (3 sites) × (5 members)
ic_data = [
    [100.0 + s * 50 + m * 10 for m in range(5)]
    for s in range(3)
]

ic = Grid(ic_data, along=[sites, members])
print(ic)
print(f"axes: {[a.name for a in ic.axes]}")
print(f"value at site=2, member=3: {ic.value_at({sites: 2, members: 3})} g C/m²")

Grid(<3 values>, along=[Axis('site', labels=['harvard_forest', 'niwot_ridge', 'metolius']), Axis('member', size=5)])
axes: ['site', 'member']
value at site=2, member=3: 230.0 g C/m²


---
## 3. EnsembleSpec

`EnsembleSpec` combines named fields into a complete ensemble specification. The total run count and iteration order are determined by the unique axes across all fields.

### Example 1: Fixed parameters + single-site climate (1 run)

In [8]:
spec = EnsembleSpec(inputs={
    "parameters": Fixed(base_params),
    "climate":    Fixed({"tair_mean": 8.5}),
    "ic":         Fixed(150.0),
})

print(spec.describe())
print()
for inputs, coord in spec.iter_runs():
    print(f"coord:  {coord}")
    print(f"inputs: {inputs}")

EnsembleSpec: 1 run
  Axes (0):
  Fields (3):
    parameters: Fixed [fixed]
    climate: Fixed [fixed]
    ic: Fixed [fixed]

coord:  {}
inputs: {'parameters': {'aMax': 11.0, 'psnTOpt': 20.0}, 'climate': {'tair_mean': 8.5}, 'ic': 150.0}


### Example 2: Multi-site run — climate and IC co-vary (zip semantics)

In [9]:
# Per-site IC values (initial plant wood C, g C/m²)
ic_per_site = [150.0, 80.0, 200.0]  # harvard_forest, niwot_ridge, metolius

# Both climate and ic use the SAME `sites` axis instance → 3 runs (zip, not 9)
spec = EnsembleSpec(inputs={
    "parameters": Fixed(base_params),
    "climate":    Grid(climate_data, along=sites),  # same `sites` object
    "ic":         Grid(ic_per_site,  along=sites),  # same `sites` object → aligned
})

print(spec.describe())
print()
for inputs, coord in spec.iter_runs():
    print(f"  {coord['site']:20s}  tair={inputs['climate']['tair_mean']}°C  ic={inputs['ic']} g C/m²")

EnsembleSpec: 3 runs
  Axes (1):
    Axis('site', labels=['harvard_forest', 'niwot_ridge', 'metolius'])
  Fields (3):
    parameters: Fixed [fixed]
    climate: Grid [site]
    ic: Grid [site]

  harvard_forest        tair=8.5°C  ic=150.0 g C/m²
  niwot_ridge           tair=1.2°C  ic=80.0 g C/m²
  metolius              tair=9.0°C  ic=200.0 g C/m²


### Example 3: Two independent axes → Cartesian product

In [10]:
# IC ensemble of 4 members, crossed with 3 sites
members4 = Axis("member", size=4)
ic_ensemble = [100.0 + m * 30 for m in range(4)]  # 100, 130, 160, 190 g C/m²

spec = EnsembleSpec(inputs={
    "parameters": Fixed(base_params),
    "climate":    Grid(climate_data, along=sites),    # varies by site
    "ic":         Grid(ic_ensemble,  along=members4), # varies by member (different axis!)
})

print(spec.describe())
print()
for inputs, coord in spec.iter_runs():
    print(f"  site={coord['site']!r:20s}  member={coord['member']}  ic={inputs['ic']} g C/m²")

EnsembleSpec: 12 runs
  Axes (2):
    Axis('site', labels=['harvard_forest', 'niwot_ridge', 'metolius'])
    Axis('member', size=4)
  Fields (3):
    parameters: Fixed [fixed]
    climate: Grid [site]
    ic: Grid [member]

  site='harvard_forest'      member=0  ic=100.0 g C/m²
  site='harvard_forest'      member=1  ic=130.0 g C/m²
  site='harvard_forest'      member=2  ic=160.0 g C/m²
  site='harvard_forest'      member=3  ic=190.0 g C/m²
  site='niwot_ridge'         member=0  ic=100.0 g C/m²
  site='niwot_ridge'         member=1  ic=130.0 g C/m²
  site='niwot_ridge'         member=2  ic=160.0 g C/m²
  site='niwot_ridge'         member=3  ic=190.0 g C/m²
  site='metolius'            member=0  ic=100.0 g C/m²
  site='metolius'            member=1  ic=130.0 g C/m²
  site='metolius'            member=2  ic=160.0 g C/m²
  site='metolius'            member=3  ic=190.0 g C/m²


### Example 4: P × S × M — three independent axes

In [11]:
param_sets_axis = Axis("param_set", size=3)
param_sets = [{"aMax": v, "psnTOpt": 20.0} for v in [9.0, 11.0, 13.0]]

spec = EnsembleSpec(inputs={
    "parameters": Grid(param_sets,   along=param_sets_axis),  # 3 param sets
    "climate":    Grid(climate_data, along=sites),             # 3 sites
    "ic":         Grid(ic_ensemble,  along=members4),          # 4 members
})

print(spec.describe())
print(f"\nTotal: {spec.n_runs} = {param_sets_axis.size} P × {sites.size} S × {members4.size} M")

EnsembleSpec: 36 runs
  Axes (3):
    Axis('param_set', size=3)
    Axis('site', labels=['harvard_forest', 'niwot_ridge', 'metolius'])
    Axis('member', size=4)
  Fields (3):
    parameters: Grid [param_set]
    climate: Grid [site]
    ic: Grid [member]

Total: 36 = 3 P × 3 S × 4 M


### Example 5: Spatially-explicit parameters

Some parameters (e.g. initial conditions) vary by *both* parameter set and site — use a multi-axis `Grid`. The shared axes prevent the combination from becoming a new independent dimension.

In [12]:
# IC varies by both param_set AND site — shape (3 P, 3 S)
ic_spatial = [
    [100.0 + p * 20 + s * 5 for s in range(3)]
    for p in range(3)
]

spec = EnsembleSpec(inputs={
    "global_params": Grid(param_sets, along=param_sets_axis),          # (P,)
    "ic":            Grid(ic_spatial, along=[param_sets_axis, sites]),  # (P, S) — shared axes
    "climate":       Grid(climate_data, along=sites),                   # (S,)
})

print(spec.describe())
print()
for inputs, coord in spec.iter_runs():
    print(
        f"  ps={coord['param_set']}  site={coord['site']!r:20s}"
        f"  aMax={inputs['global_params']['aMax']}  ic={inputs['ic']} g C/m²"
    )

EnsembleSpec: 9 runs
  Axes (2):
    Axis('param_set', size=3)
    Axis('site', labels=['harvard_forest', 'niwot_ridge', 'metolius'])
  Fields (3):
    global_params: Grid [param_set]
    ic: Grid [param_set, site]
    climate: Grid [site]

  ps=0  site='harvard_forest'      aMax=9.0  ic=100.0 g C/m²
  ps=0  site='niwot_ridge'         aMax=9.0  ic=105.0 g C/m²
  ps=0  site='metolius'            aMax=9.0  ic=110.0 g C/m²
  ps=1  site='harvard_forest'      aMax=11.0  ic=120.0 g C/m²
  ps=1  site='niwot_ridge'         aMax=11.0  ic=125.0 g C/m²
  ps=1  site='metolius'            aMax=11.0  ic=130.0 g C/m²
  ps=2  site='harvard_forest'      aMax=13.0  ic=140.0 g C/m²
  ps=2  site='niwot_ridge'         aMax=13.0  ic=145.0 g C/m²
  ps=2  site='metolius'            aMax=13.0  ic=150.0 g C/m²


`ic` uses `[param_sets_axis, sites]` — both axes already appear in other fields, so no new dimensions are added. Total: P × S = 9 runs (not P × S × S).

---
## 4. Simulating an ensemble with a toy model

`iter_runs()` produces `(inputs_dict, coordinate_dict)` pairs. Any callable can consume them.

In [13]:
def toy_model(parameters, climate, ic):
    """Trivial stand-in for a land surface model.
    
    GPP = aMax * tair_mean
    NEE = GPP - ic / 1000   (ic is initial plant wood C in g C/m²)
    """
    gpp = parameters["aMax"] * climate["tair_mean"]
    nee = gpp - ic / 1000.0
    return {"gpp": round(gpp, 3), "nee": round(nee, 3)}


# 3 param sets × 3 sites = 9 runs
spec = EnsembleSpec(inputs={
    "parameters": Grid(param_sets,   along=param_sets_axis),
    "climate":    Grid(climate_data, along=sites),
    "ic":         Fixed(150.0),
})

print(f"Running {spec.n_runs} evaluations...\n")
for inputs, coord in spec.iter_runs():
    out = toy_model(**inputs)
    print(
        f"  ps={coord['param_set']}  site={coord['site']!r:20s}"
        f"  aMax={inputs['parameters']['aMax']:5.1f}  → {out}"
    )

Running 9 evaluations...

  ps=0  site='harvard_forest'      aMax=  9.0  → {'gpp': 76.5, 'nee': 76.35}
  ps=0  site='niwot_ridge'         aMax=  9.0  → {'gpp': 10.8, 'nee': 10.65}
  ps=0  site='metolius'            aMax=  9.0  → {'gpp': 81.0, 'nee': 80.85}
  ps=1  site='harvard_forest'      aMax= 11.0  → {'gpp': 93.5, 'nee': 93.35}
  ps=1  site='niwot_ridge'         aMax= 11.0  → {'gpp': 13.2, 'nee': 13.05}
  ps=1  site='metolius'            aMax= 11.0  → {'gpp': 99.0, 'nee': 98.85}
  ps=2  site='harvard_forest'      aMax= 13.0  → {'gpp': 110.5, 'nee': 110.35}
  ps=2  site='niwot_ridge'         aMax= 13.0  → {'gpp': 15.6, 'nee': 15.45}
  ps=2  site='metolius'            aMax= 13.0  → {'gpp': 117.0, 'nee': 116.85}


---
## 5. Simple entry point: `from_runs()`

For cases where the axis algebra is overkill — just pass a flat list of input dicts.

In [14]:
spec = EnsembleSpec.from_runs(
    {"parameters": {"aMax": 9.0,  "psnTOpt": 20.0}, "climate": {"tair_mean": 5.0},  "ic": 100.0},
    {"parameters": {"aMax": 11.0, "psnTOpt": 20.0}, "climate": {"tair_mean": 8.5},  "ic": 150.0},
    {"parameters": {"aMax": 13.0, "psnTOpt": 20.0}, "climate": {"tair_mean": 10.2}, "ic": 200.0},
)

print(spec.describe())
print()
for inputs, coord in spec.iter_runs():
    out = toy_model(**inputs)
    print(f"  run={coord['run']}  aMax={inputs['parameters']['aMax']}  → {out}")

EnsembleSpec: 3 runs
  Axes (1):
    Axis('run', size=3)
  Fields (3):
    parameters: Grid [run]
    ic: Grid [run]
    climate: Grid [run]

  run=0  aMax=9.0  → {'gpp': 45.0, 'nee': 44.9}
  run=1  aMax=11.0  → {'gpp': 93.5, 'nee': 93.35}
  run=2  aMax=13.0  → {'gpp': 132.6, 'nee': 132.4}


---
## 6. Partial application: `freeze()` and `BoundSpec`

`freeze()` exposes a subset of fields as free variables, returning a `BoundSpec` callable. This is the mechanism for constructing a **parameter-to-output map** — a function that accepts only the free inputs and runs the full ensemble.

Key use case: a sampling or optimisation algorithm supplies parameter sets on the fly; the ensemble runner executes the model at all combinations with the fixed climate/IC inputs.

In [15]:
# Base spec: climate and IC are fixed across algorithm iterations; parameters will vary
base_spec = EnsembleSpec(inputs={
    "parameters": Fixed(base_params),             # placeholder — will be replaced
    "climate":    Grid(climate_data, along=sites),
    "ic":         Grid(ic_per_site,  along=sites),
})

# Designate 'parameters' as the free variable
param_map = base_spec.freeze(free=["parameters"])
print(param_map)
print(f"Free fields: {param_map.free_field_names}")

BoundSpec(free=['parameters'], fixed_fields=['climate', 'ic'])
Free fields: frozenset({'parameters'})


In [16]:
# Supply a single parameter set (auto-wrapped in Fixed) → S runs
runnable = param_map(parameters={"aMax": 15.0, "psnTOpt": 22.0})
print(runnable.describe())
print()
for inputs, coord in runnable.iter_runs():
    out = toy_model(**inputs)
    print(f"  site={coord['site']!r:20s}  ic={inputs['ic']:5.1f}  → {out}")

EnsembleSpec: 3 runs
  Axes (1):
    Axis('site', labels=['harvard_forest', 'niwot_ridge', 'metolius'])
  Fields (3):
    climate: Grid [site]
    ic: Grid [site]
    parameters: Fixed [fixed]

  site='harvard_forest'      ic=150.0  → {'gpp': 127.5, 'nee': 127.35}
  site='niwot_ridge'         ic= 80.0  → {'gpp': 18.0, 'nee': 17.92}
  site='metolius'            ic=200.0  → {'gpp': 135.0, 'nee': 134.8}


In [17]:
# Supply a Grid of parameter sets → P new axis crossed with S sites = P×S runs
p_axis = Axis("param_set", size=3)
p_grid = Grid([{"aMax": v, "psnTOpt": 20.0} for v in [9.0, 11.0, 13.0]], along=p_axis)

runnable = param_map(parameters=p_grid)
print(runnable.describe())
print()
for inputs, coord in runnable.iter_runs():
    out = toy_model(**inputs)
    print(
        f"  ps={coord['param_set']}  site={coord['site']!r:20s}"
        f"  aMax={inputs['parameters']['aMax']:5.1f}  → {out}"
    )

EnsembleSpec: 9 runs
  Axes (2):
    Axis('site', labels=['harvard_forest', 'niwot_ridge', 'metolius'])
    Axis('param_set', size=3)
  Fields (3):
    climate: Grid [site]
    ic: Grid [site]
    parameters: Grid [param_set]

  ps=0  site='harvard_forest'      aMax=  9.0  → {'gpp': 76.5, 'nee': 76.35}
  ps=1  site='harvard_forest'      aMax= 11.0  → {'gpp': 93.5, 'nee': 93.35}
  ps=2  site='harvard_forest'      aMax= 13.0  → {'gpp': 110.5, 'nee': 110.35}
  ps=0  site='niwot_ridge'         aMax=  9.0  → {'gpp': 10.8, 'nee': 10.72}
  ps=1  site='niwot_ridge'         aMax= 11.0  → {'gpp': 13.2, 'nee': 13.12}
  ps=2  site='niwot_ridge'         aMax= 13.0  → {'gpp': 15.6, 'nee': 15.52}
  ps=0  site='metolius'            aMax=  9.0  → {'gpp': 81.0, 'nee': 80.8}
  ps=1  site='metolius'            aMax= 11.0  → {'gpp': 99.0, 'nee': 98.8}
  ps=2  site='metolius'            aMax= 13.0  → {'gpp': 117.0, 'nee': 116.8}


### The pattern in an iterative algorithm

Here we simulate what it looks like when a random search calls `param_map` repeatedly, each time with a freshly proposed batch of parameter sets.

In [18]:
import random
from collections import defaultdict

random.seed(42)

best_nee, best_params = float("inf"), None

for iteration in range(4):
    # Algorithm proposes a batch of 3 parameter sets
    proposed = [{"aMax": random.uniform(5.0, 20.0), "psnTOpt": 20.0} for _ in range(3)]
    batch_axis = Axis("param_set", size=3)

    # Build and run the ensemble for this batch (S sites × 3 proposed = 9 runs)
    runnable = param_map(parameters=Grid(proposed, along=batch_axis))
    run_results = [(coord, toy_model(**inputs)) for inputs, coord in runnable.iter_runs()]

    # Aggregate: mean NEE across sites for each proposed parameter set
    nee_by_ps = defaultdict(list)
    for coord, out in run_results:
        nee_by_ps[coord["param_set"]].append(out["nee"])

    for ps_idx, nees in nee_by_ps.items():
        mean_nee = sum(nees) / len(nees)
        if mean_nee < best_nee:
            best_nee, best_params = mean_nee, proposed[ps_idx]

    print(f"Iter {iteration+1}: proposed aMax = {[round(p['aMax'], 2) for p in proposed]}")

print(f"\nBest params: {best_params}")
print(f"Best mean NEE: {best_nee:.4f}")

Iter 1: proposed aMax = [14.59, 5.38, 9.13]
Iter 2: proposed aMax = [8.35, 16.05, 15.15]
Iter 3: proposed aMax = [18.38, 6.3, 11.33]
Iter 4: proposed aMax = [5.45, 8.28, 12.58]

Best params: {'aMax': 5.375161328340004, 'psnTOpt': 20.0}
Best mean NEE: 33.3617


---
## Summary

| Concept | What it does |
|---|---|
| `Axis(name, labels=...)` | Names a dimension; identity is Python object identity |
| `Fixed(value)` | Same value every run; contributes no axis |
| `Grid(values, along=axis)` | Value indexed along one axis |
| `Grid(values, along=[ax1, ax2])` | Value indexed along multiple axes (nested list) |
| Same `Axis` instance in two fields | **Zip** — fields co-vary along that dimension |
| Different `Axis` instances | **Cross** — Cartesian product |
| `EnsembleSpec(inputs={...})` | Full spec: `.n_runs`, `.describe()`, `.iter_runs()` |
| `EnsembleSpec.from_runs(*dicts)` | Simple flat-list entry point |
| `spec.freeze(free=["field"])` | Returns a `BoundSpec` callable |
| `bound_spec(field=Grid(...))` | Binds free fields; returns a runnable `EnsembleSpec` |

**Next:** Phase 2 — `EnsembleResult`, `LocalBackend`, and `EnsembleRunner` to execute specs against real model callables in parallel.